# Pump it Up: Data Mining the Water Table

### Objetivo

El objetivo es predecir la **condición operativa de un punto de agua** (*waterpoint*) para cada registro del dataset.

### Diccionario de datos

Las variables disponibles son las siguientes:
<table>
    <thead>
        <tr>
            <th>Variable</th><th>Descripción</th>
            <th>Variable</th><th>Descripción</th>
            <th>Variable</th><th>Descripción</th>
        </tr>
    </thead>
    <tbody>
        <tr><td>amount_tsh</td><td>Altura estática total</td><td>date_recorded</td><td>Fecha de registro</td><td>funder</td><td>Financiador del pozo</td></tr>
        <tr><td>gps_height</td><td>Altitud del pozo</td><td>installer</td><td>Instalador</td><td>longitude</td><td>Longitud GPS</td></tr>
        <tr><td>latitude</td><td>Latitud GPS</td><td>wpt_name</td><td>Nombre del punto de agua</td><td>num_private</td><td>Indicador privado</td></tr>
        <tr><td>basin</td><td>Cuenca hidrográfica</td><td>subvillage</td><td>Subaldea</td><td>region</td><td>Región geográfica</td></tr>
        <tr><td>region_code</td><td>Código región</td><td>district_code</td><td>Código distrito</td><td>lga</td><td>Área gobierno local</td></tr>
        <tr><td>ward</td><td>Distrito administrativo</td><td>population</td><td>Población cercana</td><td>public_meeting</td><td>Reunión pública</td></tr>
        <tr><td>recorded_by</td><td>Registrado por</td><td>scheme_management</td><td>Administrador</td><td>scheme_name</td><td>Nombre del esquema</td></tr>
        <tr><td>permit</td><td>Tiene permiso</td><td>construction_year</td><td>Año construcción</td><td>extraction_type</td><td>Tipo extracción</td></tr>
        <tr><td>extraction_type_group</td><td>Grupo extracción</td><td>extraction_type_class</td><td>Clase extracción</td><td>management</td><td>Administración</td></tr>
        <tr><td>management_group</td><td>Grupo administración</td><td>payment</td><td>Costo agua</td><td>payment_type</td><td>Tipo pago</td></tr>
        <tr><td>water_quality</td><td>Calidad agua</td><td>quality_group</td><td>Grupo calidad</td><td>quantity</td><td>Cantidad disponible</td></tr>
        <tr><td>quantity_group</td><td>Grupo cantidad</td><td>source</td><td>Fuente agua</td><td>source_type</td><td>Tipo fuente</td></tr>
        <tr><td>source_class</td><td>Clase fuente</td><td>waterpoint_type</td><td>Tipo punto agua</td><td>waterpoint_type_group</td><td>Grupo tipo punto</td></tr>
    </tbody>
</table>

**Variable objetivo (Target) del Dataset**

Las etiquetas objetivo son simples y tienen **tres posibles valores**:

- **functional**: El punto de agua está operativo y no requiere reparaciones.
- **functional needs repair**: El punto de agua está operativo, pero necesita reparaciones.
- **non functional**: El punto de agua no está operativo.

**Dataset obtenido de la competencia de DrivenData**:
[Pump It Up: Data Mining the Water Table](https://www.drivendata.org/competitions/7/pump-it-up-data-mining-the-water-table/)

*Los comentarios en azul representan el registro de mi proceso de pensamiento durante el desarrollo del proyecto.*
*Los markdowns normales corresponden a la construcción del modelo.*

### Librerías

In [28]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from xgboost import XGBClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
from category_encoders import TargetEncoder

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import warnings
warnings.filterwarnings('ignore')

### Carga de Datos

Se realiza la carga de datos, búsqueda de duplicados y unión de DataFrames.

In [29]:
# Carga de Datos
train = pd.read_csv("train.csv")
labels = pd.read_csv("labels.csv")
test  = pd.read_csv("test.csv")

In [30]:
# Búsqueda de duplicados
print(train.id.nunique())
print(labels.id.nunique())

59400
59400


No se encuentran ids duplicadas.

In [31]:
# Unión de datos por 'id'
train_merge = pd.merge(train, labels, on='id')

# Crear columna para diferenciarlos
train_merge['dataset'] = 'train'
test['dataset'] = 'test'

# Concatenar en un único DataFrame
df = pd.concat([train_merge, test], axis=0, ignore_index=True)

# Dimensiones del DataFrame
print(f"Dimensión del DataFrame: {df.shape}")
df.head()

Dimensión del DataFrame: (74250, 42)


,id,amount_tsh,date_recorded,funder,gps_height,installer,longitude,latitude,wpt_name,num_private,...,quality_group,quantity,quantity_group,source,source_type,source_class,waterpoint_type,waterpoint_type_group,status_group,dataset
0,69572,6000.0,2011-03-14,Roman,1390,Roman,34.938093,-9.856322,none,0,...,good,enough,enough,spring,spring,groundwater,communal standpipe,communal standpipe,functional,train
1,8776,0.0,2013-03-06,Grumeti,1399,GRUMETI,34.698766,-2.147466,Zahanati,0,...,good,insufficient,insufficient,rainwater harvesting,rainwater harvesting,surface,communal standpipe,communal standpipe,functional,train
2,34310,25.0,2013-02-25,Lottery Club,686,World vision,37.460664,-3.821329,Kwa Mahundi,0,...,good,enough,enough,dam,dam,surface,communal standpipe multiple,communal standpipe,functional,train
3,67743,0.0,2013-01-28,Unicef,263,UNICEF,38.486161,-11.155298,Zahanati Ya Nanyumbu,0,...,good,dry,dry,machine dbh,borehole,groundwater,communal standpipe multiple,communal standpipe,non functional,train
4,19728,0.0,2011-07-13,Action In A,0,Artisan,31.130847,-1.825359,Shuleni,0,...,good,seasonal,seasonal,rainwater harvesting,rainwater harvesting,surface,communal standpipe,communal standpipe,functional,train


Esta inspección inicial permite verificar que la unión se realizó correctamente y revisar la estructura general de los datos antes del preprocesamiento.

### Exploración

Revisión inicial de tipos de datos, columnas redundantes y distribución de categorías.

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74250 entries, 0 to 74249
Data columns (total 42 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     74250 non-null  int64  
 1   amount_tsh             74250 non-null  float64
 2   date_recorded          74250 non-null  object 
 3   funder                 69743 non-null  object 
 4   gps_height             74250 non-null  int64  
 5   installer              69718 non-null  object 
 6   longitude              74250 non-null  float64
 7   latitude               74250 non-null  float64
 8   wpt_name               74248 non-null  object 
 9   num_private            74250 non-null  int64  
 10  basin                  74250 non-null  object 
 11  subvillage             73780 non-null  object 
 12  region                 74250 non-null  object 
 13  region_code            74250 non-null  int64  
 14  district_code          74250 non-null  int64  
 15  lg

Existen variables que aparentan tener la misma información. La eliminación de ellas se hará en el paso de limpieza.

In [33]:
# Variables aparentemente duplicadas
redundant_columns = [
    'scheme_management', 'scheme_name',
    'management', 'management_group',
    'extraction_type', 'extraction_type_group', 'extraction_type_class',
    'water_quality', 'quality_group',
    'quantity', 'quantity_group',
    'source', 'source_type', 'source_class',
    'payment', 'payment_type',
    'waterpoint_type', 'waterpoint_type_group'
]

df[redundant_columns].nunique()

# Ver las distintas categorías
for var in redundant_columns:
    print(f"\nVariable: {var}")
    print(df[df['dataset'] == 'train'][var].value_counts(dropna=False))


Variable: scheme_management
scheme_management
VWC                 36793
WUG                  5206
NaN                  3878
Water authority      3153
WUA                  2883
Water Board          2748
Parastatal           1680
Private operator     1063
Company              1061
Other                 766
SWC                    97
Trust                  72
Name: count, dtype: int64

Variable: scheme_name
scheme_name
NaN                     28810
K                         682
Borehole                  546
Chalinze wate             405
M                         400
                        ...  
Mradi wa maji Vijini        1
Villagers                   1
Magundi water supply        1
Saadani Chumv               1
Mtawanya                    1
Name: count, Length: 2696, dtype: int64

Variable: management
management
vwc                 40507
wug                  6515
water board          2933
wua                  2535
private operator     1971
parastatal           1768
water authority      

Se observan múltiples variables que representan la misma información en distintos niveles de agregación, por lo que se eliminan las redundantes para reducir dimensionalidad y evitar multicolinealidad.

Se conservan las siguientes variables por ofrecer el mejor equilibrio entre detalle y generalización:

Se conservan las variables: `extraction_type_class`, `management_group`, `payment_type`, `quality_group`, `quantity`, `scheme_management`, `source_type`, `waterpoint_type_group`.

### Limpieza

Se eliminan las columnas redundantes o sin valor informativo y se tratan los valores nulos.

In [34]:
# Eliminar columnas redundantes e innecesarias
columns_drop = [
    'scheme_name', 'management', 'extraction_type', 'extraction_type_group',
    'water_quality', 'quantity_group', 'source', 'source_class',
    'payment', 'waterpoint_type',
    'num_private', 'wpt_name', 'recorded_by', 'subvillage',
]
df.drop(columns=columns_drop, inplace=True, errors='ignore')

In [35]:
# Exploración de categorías y valores de las columnas restantes
for i in df.columns:
    print(f"\nVariable: {i}")
    print(df[i].value_counts())


Variable: id
id
69572    1
51488    1
40697    1
40221    1
68749    1
        ..
66431    1
70261    1
26783    1
46768    1
68707    1
Name: count, Length: 74250, dtype: int64

Variable: amount_tsh
amount_tsh
0.0         52049
500.0        3874
50.0         3103
1000.0       1858
20.0         1812
            ...  
250000.0        1
350000.0        1
5400.0          1
53.0            1
0.5             1
Name: count, Length: 102, dtype: int64

Variable: date_recorded
date_recorded
2011-03-17    695
2011-03-15    694
2013-02-03    684
2011-03-16    650
2011-03-14    649
             ... 
2011-09-01      1
2011-09-28      1
2011-09-16      1
2011-09-13      1
2004-01-09      1
Name: count, Length: 369, dtype: int64

Variable: funder
funder
Government Of Tanzania    11299
Danida                     3907
Hesawa                     2782
Rwssp                      1703
World Bank                 1701
                          ...  
Sido                          1
Mosqure                   

- **id**: solo identificador único del registro, no aporta información predictiva. 
- **amount_tsh**: variable altamente sesgada con muchos ceros, indica tarifa pero con poca variabilidad útil.
- **date_recorded**: variable temporal que se puede transformar en features (año, mes), no se usa en crudo.
- **funder**: alta cardinalidad, muchos valores únicos.
- **gps_height**: variable útil geográficamente, pero con muchos ceros (posibles faltantes).
- **installer**: alta cardinalidad y ruido en categorías.
- **longitude / latitude**: variables geográficas clave, mantener.
- **basin / region / region_code / district_code / lga / ward**: variables jerárquicas geográficas, algunas redundantes entre sí.
- **population**: muchos ceros y valores extremos 
- **public_meeting / permit**: variables booleanas útiles, mantener.
- **construction_year**: muchos ceros (missing implícito), se transforma a edad.
- **extraction_type_class**: versión resumida y útil del tipo de extracción.
- **management_group**: versión agrupada de gestión, más estable.
- **payment_type**: versión limpia y agrupada de pagos.
- **quality_group**: resumen de calidad del agua, reduce ruido.
- **quantity**: variable importante sobre disponibilidad de agua.
- **source_type**: nivel intermedio adecuado de fuente de agua.
- **waterpoint_type_group**: agrupación útil del tipo de punto de agua.
- **status_group**: variable objetivo (target).
- **dataset**: indicador train/test, solo para control del split.

In [36]:
# Total de NaN por variable
df.isnull().sum()

id                           0
amount_tsh                   0
date_recorded                0
funder                    4507
gps_height                   0
installer                 4532
longitude                    0
latitude                     0
basin                        0
region                       0
region_code                  0
district_code                0
lga                          0
ward                         0
population                   0
public_meeting            4155
scheme_management         4847
permit                    3793
construction_year            0
extraction_type_class        0
management_group             0
payment_type                 0
quality_group                0
quantity                     0
source_type                  0
waterpoint_type_group        0
status_group             14850
dataset                      0
dtype: int64

Las variables `scheme_management`, `installer` y `funder` presentan valores nulos que se imputan creando una nueva categoría denominada `'unknown'`, ya que la ausencia de información en estas variables categóricas puede representar una categoría informativa por sí misma.

En el caso de `public_meeting` y `permit`, los valores nulos se imputan con la moda.

La variable `status_group` (target) mantiene sus valores nulos en el conjunto de test, ya que precisamente es la variable que se busca predecir.

In [37]:
# Tratamiento de NaNs
df['scheme_management'].fillna('unknown', inplace=True)
df['installer'].fillna('unknown', inplace=True)
df['funder'].fillna('unknown', inplace=True)
df['public_meeting'].fillna(True, inplace=True)
df['permit'].fillna(True, inplace=True)
# df.isnull().sum()  # Revisión post-imputación

Se imputaron los valores ausentes.

### Feature Engineering

Se crean nuevas variables a partir de la información existente para mejorar la capacidad predictiva del modelo. El proceso se divide en tres bloques: variables temporales, variables geoespaciales y variables binarias/transformaciones.

#### 1 Features Temporales

In [38]:
# Convertir date_recorded a datetime y extraer componentes
df['date_recorded'] = pd.to_datetime(df['date_recorded'])
df['year_recorded']  = df['date_recorded'].dt.year
df['month_recorded'] = df['date_recorded'].dt.month

# Calcular EDAD DE LA BOMBA
# Año de construcción 0 = desconocido, se trata como NaN
df['construction_year_clean'] = df['construction_year'].replace(0, np.nan)
df['pump_age'] = df['year_recorded'] - df['construction_year_clean']
# Las bombas más viejas tienen mayor probabilidad de fallo

# Tramificación de la edad
df['pump_age_bin'] = pd.cut(
    df['pump_age'],
    bins=[-1, 5, 10, 20, 30, 100],
    labels=['0-5', '6-10', '11-20', '21-30', '30+']
)

print('Distribución de edad de bombas:')
print(df[df['dataset'] == 'train']['pump_age_bin'].value_counts())

# Eliminar date_recorded: ya no es necesaria tras extraer las features
df.drop(columns=['date_recorded'], inplace=True, errors='ignore')

Distribución de edad de bombas:
pump_age_bin
0-5      11630
11-20     9456
30+       6336
21-30     5633
6-10      5627
Name: count, dtype: int64


A partir de `date_recorded` se ha extraído el año y mes de registro para incorporar la antigüedad como característica. La edad de la bomba (`pump_age`) se calcula restando el año de construcción al año de registro, tratando los valores `0` en `construction_year` como desconocidos (NaN). Se espera que las bombas más antiguas presenten mayor probabilidad de fallo.

La edad se agrupa además en intervalos (`pump_age_bin`) para capturar la relación por rangos de antigüedad.

#### 2 Features Geoespaciales

In [39]:
# Centro aproximado de Tanzania (Dodoma)
TANZANIA_CENTER_LAT = -6.1722
TANZANIA_CENTER_LON = 35.7395

# Distancia euclidiana al centro del país
df['dist_to_center'] = np.sqrt(
    (df['latitude']  - TANZANIA_CENTER_LAT)**2 +
    (df['longitude'] - TANZANIA_CENTER_LON)**2
)

# Interacción lat × lon como feature compuesta
df['lat_lon_product'] = df['latitude'] * df['longitude']

# Densidad regional de bombas (cuántas bombas hay en la misma región)
region_density = df.groupby('region')['id'].transform('count')
df['region_pump_density'] = region_density

# Altitud al cuadrado (captura relación no lineal con la altitud)
df['gps_height_sq'] = df['gps_height'] ** 2

Se crean nuevas variables para mejorar la capacidad predictiva del modelo capturando patrones geográficos y no lineales:

`dist_to_center`: mide la distancia de cada punto al centro aproximado de Tanzania (Dodoma), capturando dependencia espacial global.      
`lat_lon_product`: interacción entre latitud y longitud para capturar relaciones espaciales no lineales.      
`region_pump_density`: densidad de bombas por región, útil para reflejar presión de demanda e infraestructura existente.    
`ps_height_sq`: transformación cuadrática de la altitud para modelar efectos no lineales de la elevación.       

Otro cambio importante para mejorar el modelo.

#### 3 Features Binarias y Transformaciones

In [40]:
# Indicadores binarios de presencia de información relevante
df['amount_tsh_bin']        = (df['amount_tsh'] > 0).astype(int)
df['has_construction_year'] = (df['construction_year'] > 0).astype(int)
df['has_population']        = (df['population'] > 0).astype(int)

# Transformación logarítmica de amount_tsh para reducir el efecto de outliers
df['amount_tsh_log'] = np.log1p(df['amount_tsh'])

Variable: Feature Engineering (binaria y transformación)

Se crean nuevas variables para mejorar la calidad de la información y reducir problemas de escala y sesgo:

`amount_tsh_bin`: indica si existe o no pago de agua (> 0). Captura la diferencia entre pozos con y sin coste.       
`has_construction_year`: indica si el año de construcción es válido (> 0), ya que 0 representa dato faltante.       
`has_population`: indica si existe registro de población (> 0), útil para distinguir datos faltantes reales.       

`amount_tsh_log`: transformación logarítmica de la tarifa de agua para reducir el impacto de outliers y sesgo extremo en la distribución.        

Estas variables permiten al modelo manejar mejor valores ausentes implícitos, asimetrías y outliers.

### Target Encoding

Se aplica Target Encoding a las variables categóricas de alta cardinalidad. Para evitar **data leakage**, se separan primero los conjuntos train y test, y el encoder se ajusta exclusivamente sobre train.

In [41]:
# Separar train y test ANTES del encoding para evitar leakage
df_train = df[df['dataset'] == 'train'].copy()
df_test  = df[df['dataset'] == 'test'].copy()

# Encodear la target para usarla en target encoding
le = LabelEncoder()
df_train['status_group_enc'] = le.fit_transform(df_train['status_group'])

# Variables candidatas a Target Encoding (alta cardinalidad)
target_enc_cols = [
    'installer', 'funder', 'ward', 'lga',
    'scheme_management', 'extraction_type_class',
    'payment_type', 'source_type', 'basin', 'region'
]

# Target Encoder con smoothing para evitar overfitting en categorías raras
te = TargetEncoder(cols=target_enc_cols, smoothing=10)

# Fit sobre train
df_train[target_enc_cols] = te.fit_transform(
    df_train[target_enc_cols],
    df_train['status_group_enc']
)

# Aplicar a test (solo transform, sin refit)
df_test[target_enc_cols] = te.transform(df_test[target_enc_cols])

print('Target Encoding aplicado a:', target_enc_cols)

Target Encoding aplicado a: ['installer', 'funder', 'ward', 'lga', 'scheme_management', 'extraction_type_class', 'payment_type', 'source_type', 'basin', 'region']


Con  **Target Encoding** se convierten  variables categóricas de alta cardinalidad en valores numéricos basados en la media de la variable objetivo.

Proceso:

- Se separan los datos en **train y test** para evitar data leakage.
- Se codifica la variable objetivo `status_group` a valores numéricos.
- Se seleccionan variables categóricas con alta cardinalidad.
- Para cada categoría, se calcula la media del target en el conjunto de entrenamiento.
- Se reemplazan las categorías por ese valor promedio.
- En test solo se aplica la transformación aprendida en train.

Se utiliza **smoothing** para evitar sobreajuste en categorías poco frecuentes.

Resultado: variables categóricas convertidas en variables numéricas informativas para el modelo.

### Preparar X_train, y_train, X_test

Se preparan las matrices de entrada al modelo: codificación de variables de baja cardinalidad con One-Hot Encoding, alineación de columnas entre train y test, y definición de las variables finales.

In [42]:
# Variables a eliminar del modelo
drop_cols = ['dataset', 'status_group', 'status_group_enc', 'id',
             'construction_year',   # Reemplazada por pump_age
             'pump_age_bin']        # Se usa pump_age numérico

# Convertir booleanos a int
for col in df_train.select_dtypes(include='bool').columns:
    df_train[col] = df_train[col].astype(int)
    df_test[col]  = df_test[col].astype(int)

# One-Hot Encoding para variables de baja cardinalidad
low_card_cols = ['quality_group', 'quantity', 'waterpoint_type_group', 'management_group']
df_train = pd.get_dummies(df_train, columns=[c for c in low_card_cols if c in df_train.columns])
df_test  = pd.get_dummies(df_test,  columns=[c for c in low_card_cols if c in df_test.columns])

# Alinear columnas train/test (por si get_dummies genera columnas distintas)
df_train, df_test = df_train.align(df_test, join='left', axis=1, fill_value=0)

# Definir X e y
feature_cols = [c for c in df_train.columns if c not in drop_cols]
# Eliminar columnas que aún sean object
feature_cols = [c for c in feature_cols if df_train[c].dtype != 'object']

X            = df_train[feature_cols].fillna(-999)
y            = le.fit_transform(df_train['status_group'])
X_test_final = df_test[feature_cols].fillna(-999)

print(f'Shape X: {X.shape} | Shape X_test: {X_test_final.shape}')
print('Clases:', le.classes_)

Shape X: (59400, 53) | Shape X_test: (14850, 53)
Clases: ['functional' 'functional needs repair' 'non functional']


Se realiza el preprocesamiento final antes del entrenamiento del modelo:

- Se eliminan columnas no útiles o redundantes (`id`, variables originales ya transformadas, y el target en crudo).
- Se convierten variables booleanas a formato numérico (0/1).
- Se aplica **One-Hot Encoding** a variables categóricas de baja cardinalidad para evitar imposición de orden.
- Se alinean las columnas de train y test para asegurar consistencia entre ambos conjuntos.
- Se definen:
  - **X**: variables predictoras
  - **y**: variable objetivo codificada
  - **X_test_final**: conjunto de prueba listo para predicción
- Se rellenan valores faltantes con -999 para permitir el uso en modelos basados en árboles.

Lo que nos lleva a tener un dataset completamente numérico, limpio y listo para entrenamiento.

### LightGBM con Optuna

Se optimizan los hiperparámetros del modelo LightGBM mediante búsqueda bayesiana con Optuna (50 trials).

In [43]:
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def objective_lgbm(trial):
    params = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'n_estimators':      trial.suggest_int('n_estimators', 300, 1000),
        'max_depth':         trial.suggest_int('max_depth', 4, 12),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 31, 255),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }
    model = lgb.LGBMClassifier(**params, random_state=42)
    model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    preds = model.predict(X_val)
    return accuracy_score(y_val, preds)

# 50 trials — tarda ~10-15 min
study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(objective_lgbm, n_trials=50, show_progress_bar=True)

print(f'\nMejor accuracy LGBM: {study_lgbm.best_value:.4f}')
print(f'Mejores parámetros: {study_lgbm.best_params}')

  0%|          | 0/50 [00:00<?, ?it/s]


Mejor accuracy LGBM: 0.8189
Mejores parámetros: {'n_estimators': 920, 'max_depth': 8, 'learning_rate': 0.04141240158548312, 'num_leaves': 80, 'min_child_samples': 26, 'subsample': 0.8403516273965683, 'colsample_bytree': 0.6456485963913453, 'reg_alpha': 0.027479202747828498, 'reg_lambda': 2.0273328683251975e-07}


Se entrena un modelo de clasificación multiclase utilizando **LightGBM** con optimización de hiperparámetros mediante **Optuna**:

- Se divide el dataset en entrenamiento y validación (80/20) con estratificación para mantener la distribución de clases.
- Se define una función objetivo que:
  - Entrena un modelo LightGBM.
  - Evalúa el desempeño con **accuracy** en validación.
- Optuna explora diferentes combinaciones de hiperparámetros (profundidad, learning rate, regularización, etc.).
- Se aplican técnicas de control de overfitting como:
  - early stopping
  - subsampling de features y muestras
- Se ejecutan 50 pruebas para encontrar la mejor configuración.

Resultado:
- Se obtiene un modelo optimizado con accuracy ~0.82 en validación.
- Se seleccionan los mejores hiperparámetros para el modelo final.

In [44]:
# Entrenar el mejor modelo LGBM sobre TODOS los datos de train
best_lgbm = lgb.LGBMClassifier(**study_lgbm.best_params, random_state=42, verbosity=-1)
best_lgbm.fit(X, y)

# Validación cruzada para ver score real
cv_lgbm = cross_val_score(best_lgbm, X, y, cv=5, scoring='accuracy')
print(f'CV LGBM: {cv_lgbm.mean():.4f} ± {cv_lgbm.std():.4f}')

CV LGBM: 0.8186 ± 0.0022


Se entrena el modelo final utilizando los mejores hiperparámetros encontrados:

- Se reentrena **LightGBM** con todo el conjunto de entrenamiento.
- Se evalúa el rendimiento mediante **validación cruzada (5-fold)** para obtener una estimación más robusta.
- Se utiliza accuracy como métrica principal.

Resultado:
- Accuracy promedio CV: **0.817**
- Desviación estándar baja (~0.002), indicando estabilidad del modelo.

Este paso confirma que el modelo generaliza de forma consistente antes de generar predicciones finales.

In [45]:
# Evaluar LGBM en validación
y_val_pred_lgbm = best_lgbm.predict(X_val)
print('LGBM')
print(classification_report(y_val, y_val_pred_lgbm, target_names=le.classes_))

LGBM
                         precision    recall  f1-score   support

             functional       0.88      0.96      0.92      6452
functional needs repair       0.86      0.59      0.70       863
         non functional       0.94      0.87      0.90      4565

               accuracy                           0.90     11880
              macro avg       0.89      0.81      0.84     11880
           weighted avg       0.90      0.90      0.90     11880



El modelo **LightGBM** alcanza una **accuracy de 0.89** en el conjunto de validación, lo que refleja un buen rendimiento general. Presenta un desempeño sólido en la clase *functional*, con alta capacidad de detección, y también en *non functional*, con buenos niveles de precisión y recall. La principal debilidad está en la clase *functional needs repair*, donde el recall es de 0.55, indicando dificultades para identificar correctamente esta categoría intermedia. En general, el modelo funciona bien, pero requiere mejoras en la clase minoritaria.

#### Predicciones LGBM

In [46]:
# Mapeo de índices a etiquetas
label_mapping = {0: 'functional', 1: 'functional needs repair', 2: 'non functional'}

predictions_lgbm = best_lgbm.predict(X_test_final)

predicciones_lgbm = pd.DataFrame({
    'id': df_test['id'],
    'status_group': [label_mapping[p] for p in predictions_lgbm]
})
print(predicciones_lgbm.shape)

# Guardar el archivo
predicciones_lgbm.to_csv('predicciones_lgbm.csv', index=False, encoding='utf-8')
print('\nArchivo guardado: predicciones_lgbm.csv')
print(predicciones_lgbm['status_group'].value_counts())

(14850, 2)

Archivo guardado: predicciones_lgbm.csv
status_group
functional                 8955
non functional             5315
functional needs repair     580
Name: count, dtype: int64


Se generan las predicciones finales utilizando el modelo entrenado:

- Se aplica el modelo **LightGBM** sobre el conjunto de test (`X_test_final`).
- Se convierte la salida numérica del modelo a las etiquetas originales mediante un mapeo.
- Se construye un DataFrame con:
  - `id`: identificador del registro
  - `status_group`: predicción final de la condición del pozo

Resultado:
- Se obtiene el archivo de predicciones listo para entrega o evaluación.
- Se verifica la dimensión del output para confirmar consistencia con el dataset de test.

### CatBoost con Optuna

Se optimiza un modelo CatBoost y se combina con LGBM mediante promedio de probabilidades (ensemble).

In [47]:
def objective_catboost(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 1500),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'random_strength': trial.suggest_float('random_strength', 1e-9, 10, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        'loss_function': 'MultiClass',
        'eval_metric': 'Accuracy',
        'verbose': False,
        'random_seed': 42
    }
    
    model = CatBoostClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=50, verbose=False)
    
    preds = model.predict(X_val)
    return accuracy_score(y_val, preds)

# Ejecutar optimización
study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_catboost, n_trials=30, show_progress_bar=True)

print(f'\nMejor accuracy CatBoost: {study_cat.best_value:.4f}')

  0%|          | 0/30 [00:00<?, ?it/s]


Mejor accuracy CatBoost: 0.8170


Se entrena un modelo **CatBoost** utilizando optimización de hiperparámetros con **Optuna**:

- Se define una función objetivo que entrena el modelo y evalúa su desempeño en validación.
- Se ajustan hiperparámetros clave como profundidad, learning rate, regularización y muestreo.
- Se utiliza **early stopping** para evitar overfitting.
- Se maximiza la métrica de **accuracy**.

Resultado:
- Se ejecutan 30 trials de optimización.
- Se obtiene un mejor accuracy de aproximadamente **0.815** en validación.

Este modelo se utiliza como alternativa al LightGBM para comparación de rendimiento.

In [54]:
# Mapeo de índices a etiquetas
label_mapping = {0: 'functional', 1: 'functional needs repair', 2: 'non functional'}

# Predicciones CatBoost
predictions_cat = best_cat.predict(X_test_final).ravel().astype(int)

# DataFrame de submission
predicciones_cat = pd.DataFrame({
    'id': df_test['id'],
    'status_group': [label_mapping[p] for p in predictions_cat]
})

print(predicciones_cat.shape)

# Guardar archivo
predicciones_cat.to_csv('predicciones_cat.csv', index=False, encoding='utf-8')

print('\nArchivo guardado: predicciones_cat.csv')
print(predicciones_cat['status_group'].value_counts())

(14850, 2)

Archivo guardado: predicciones_cat.csv
status_group
functional                 8960
non functional             5329
functional needs repair     561
Name: count, dtype: int64


Se entrena el modelo final **CatBoost** utilizando los mejores hiperparámetros obtenidos mediante Optuna y todo el conjunto de entrenamiento:

- Se reentrena el modelo con **X y y completos** para maximizar el aprendizaje.
- Se generan predicciones sobre el conjunto de test.
- Se convierten las predicciones numéricas a etiquetas originales mediante el mapeo definido.
- Se construye el archivo de submission con:
  - `id`: identificador del registro
  - `status_group`: predicción final

Finalmente, se exporta el archivo en formato CSV listo para su evaluación en la competencia.

In [53]:
# Evaluar CatBoost solo en validación
y_val_pred_cat = best_cat.predict(X_val)

print('CatBoost')
print(classification_report(y_val, y_val_pred_cat, target_names=le.classes_))

CatBoost
                         precision    recall  f1-score   support

             functional       0.87      0.95      0.91      6452
functional needs repair       0.84      0.54      0.66       863
         non functional       0.92      0.86      0.89      4565

               accuracy                           0.89     11880
              macro avg       0.88      0.78      0.82     11880
           weighted avg       0.89      0.89      0.88     11880



El modelo **CatBoost** alcanza una **accuracy de 0.88** en el conjunto de validación, mostrando un buen desempeño general, ligeramente inferior al de LightGBM. En la clase *functional* presenta un rendimiento sólido, con alta capacidad de detección, y en *non functional* también mantiene buenos niveles de precisión y recall. La principal debilidad nuevamente está en la clase *functional needs repair*, con un recall de 0.51, lo que indica dificultades para identificar correctamente esta categoría intermedia. En conjunto, es un modelo competitivo, pero con menor sensibilidad en la clase minoritaria.

#### Comparación de modelos (LightGBM vs CatBoost)

Ambos modelos muestran un rendimiento muy similar, con **LightGBM ligeramente superior (0.89 accuracy)** frente a **CatBoost (0.88 accuracy)**. En la clase *functional*, el desempeño es prácticamente idéntico, con un recall de 0.95 en ambos casos, lo que refleja una excelente detección de pozos funcionales. En *non functional*, LightGBM tiene una ligera ventaja en precisión y balance general. La mayor diferencia se observa en *functional needs repair*, donde ambos modelos tienen dificultades, pero LightGBM obtiene un recall ligeramente mayor (0.55 vs 0.51), siendo más sensible a esta clase minoritaria. En general, **LightGBM resulta ligeramente superior**, aunque ambos modelos son altamente competitivos.